# 02 — Exploratory Data Analysis (EDA)

## Syfte
Detta steg motsvarar kursens krav på **EDA/statistik**. Innan vi utforskar
mönster i datasetet behöver vi dock först slutföra ett bitar av
dataförberedelsen som identifierades i steg 01: konvertering av
`dob_matlab` till en riktig `age`-kolumn.

Notebooken körs i en egen kernel, separat från `01_data_preparation.ipynb`,
så vi laddar in `df` på nytt här via samma `src/data_loader.py`-funktion
som användes i steg 01 — inga globala variabler delas mellan notebooks.

In [1]:
import sys
sys.path.append("..")  # så att src/ blir importerbar från notebooks/-mappen

from src.data_loader import load_wiki_mat

df = load_wiki_mat("../data/raw/wiki_crop/wiki.mat")
df.head()

,full_path,dob_matlab,photo_taken,gender,face_score,second_face_score,name,face_location
0,17/10000217_1981-05-05_2009.jpg,723671,2009,1.0,4.300962,NaN,Sami Jauhojärvi,"[111.29109473290997, 111.29109473290997, 252.6..."
1,48/10000548_1925-04-04_1964.jpg,703186,1964,1.0,2.645639,1.949248,Dettmar Cramer,"[252.48330229530742, 126.68165114765371, 354.5..."
2,12/100012_1948-07-03_2008.jpg,711677,2008,1.0,4.329329,NaN,Marc Okrand,"[113.52, 169.83999999999997, 366.08, 422.4]"
3,65/10001965_1930-05-23_1961.jpg,705061,1961,1.0,-inf,NaN,Aleksandar Matanović,"[1, 1, 634, 440]"
4,16/10002116_1971-05-31_2012.jpg,720044,2012,0.0,3.408442,NaN,Diana Damrau,"[171.61031405173117, 75.57451239763239, 266.76..."


## Test av datumkonvertering

Innan vi beräknar ålder över hela datasetet, testar vi
`matlab_datenum_to_datetime` (definierad i `src/preprocessing.py`) på ett
litet stickprov för att bekräfta att konverteringen ger rimliga datum
(inga orimliga år som 0001 eller 3000).

In [2]:
from src.preprocessing import matlab_datenum_to_datetime

sample = df["dob_matlab"].dropna().sample(5, random_state=42)

for val in sample:
    print(val, "->", matlab_datenum_to_datetime(val))

713150 -> 1952-07-15 00:00:00
699769 -> 1915-11-26 00:00:00
727079 -> 1990-09-03 00:00:00
723288 -> 1980-04-17 00:00:00
727078 -> 1990-09-02 00:00:00


Datumen ovan (1915–1990) ser rimliga ut — inga extremvärden. Vi går
vidare och beräknar `age`-kolumnen över hela datasetet med `compute_age`.

In [3]:
from src.preprocessing import compute_age

df = compute_age(df)
df[["dob_matlab", "photo_taken", "age"]].sample(10, random_state=42)

,dob_matlab,photo_taken,age
1003,713150,2013,61
15712,699769,1943,28
26837,727079,2014,24
50657,723288,2006,26
16974,727078,2015,25
39810,723926,2013,31
21014,704092,1954,27
24108,721889,2013,37
52095,727680,2014,22
53091,720285,1970,-2


### Ålderberäkning: initial kontroll

`compute_age()` beräknar ålder som `photo_taken - födelseår` (extraherat från `dob_matlab`).

Ett stickprov på 10 slumpade rader visar i huvudsak rimliga åldrar (22–61 år), men rad 53091 
ger `age = -2` — dvs. fotot är daterat *före* personens födelse. Detta är ett känt problem i 
IMDB-WIKI-datasetet: metadata är skrapad från Wikipedia och namn-till-person-matchningen är 
inte alltid korrekt, vilket kan resultera i felaktiga födelsedatum för den avbildade personen.

Detta hanteras inte här utan flaggas för steg 3 (filtrering av orimliga åldrar), där vi 
definierar rimliga gränser (t.ex. 0–100 år) och undersöker omfattningen av problemet över 
hela datasetet, inte bara i detta stickprov.

## Ansiktsdetektering: kartläggning av saknade ansikten

`face_score == -inf` betyder att ansiktsdetektorn inte hittade något
ansikte i bilden. Vi flaggar detta med `flag_face_detection()` (utan att
ta bort rader ännu) för att kunna redovisa omfattningen här i EDA:n.

In [4]:
from src.preprocessing import flag_face_detection

df = flag_face_detection(df)
n_missing = (~df["face_detected"]).sum()
pct_missing = n_missing / len(df) * 100

print(f"Rader utan detekterat ansikte: {n_missing} av {len(df)} ({pct_missing:.2f}%)")

Rader utan detekterat ansikte: 18016 av 62328 (28.91%)


### Ansiktsdetektering: resultat

**18 016 av 62 328 rader (28,91%)** saknar ett detekterat ansikte
(`face_score == -inf`). Detta är en betydligt högre andel än förväntat,
men stämmer med ett känt problem i IMDB-WIKI: `wiki_crop`-bilderna är
beskurna utifrån Wikipedia-metadata om var en person *förväntas* synas,
inte utifrån en verifierad ansiktsdetektion. Ansiktsdetektorn som
genererade `face_score` misslyckas ofta på äldre/lågupplösta foton eller
där bild-till-person-matchningen från Wikipedia var felaktig från början.

Detta flaggas här (`face_detected`-kolumnen) men filtreras INTE bort i
detta steg — se `filter_valid_faces()` i `src/preprocessing.py`, som
anropas senare i pipelinen (precis innan embedding-extraktion), i linje
med principen att inte kasta bort data i tidiga, generella
förberedelsesteg.

**Konsekvens för projektet:** av de ursprungliga 62 328 raderna kommer
sannolikt endast ~44 312 (71,09%) att vara användbara för embeddings och
nedströms klassificering. Detta bortfall bör redovisas explicit i
slutrapporten som en del av dataförberedelse-motiveringen.

## Gender: kartläggning av saknade värden

`gender` saknas (NaN) för en delmängd av raderna. Detta är äkta,
icke-imputerbar saknad data — vi flaggar den med `flag_missing_gender()`
istället för att gissa eller ta bort rader här.

In [5]:
from src.preprocessing import flag_missing_gender

df = flag_missing_gender(df)
n_missing = df["gender_missing"].sum()
pct_missing = n_missing / len(df) * 100

print(f"Rader utan gender: {n_missing} av {len(df)} ({pct_missing:.2f}%)")

Rader utan gender: 2643 av 62328 (4.24%)


### Gender: resultat

**2 643 av 62 328 rader (4,24%)** saknar `gender`. Detta matchar exakt
den siffra som identifierades redan i steg 01 (`01_data_preparation.ipynb`),
vilket bekräftar att ingen oavsiktlig dataförlust eller förändring skett
mellan stegen.

Denna brist är äkta och icke-imputerbar — det finns ingen tillförlitlig
metod att gissa kön utifrån tillgängliga features, så vi flaggar
raderna (`gender_missing`-kolumnen) utan att imputera eller ta bort dem
här. Filtrering sker separat (`filter_valid_gender()`) och tillämpas
endast där `gender` faktiskt används som målvariabel, t.ex. vid träning
av ålder-/kön-estimeringsmodellen — inte i pipelinens generella
förberedelsesteg.

## second_face_score: kartläggning av flera ansikten i bild

`second_face_score` är NaN när inget andra ansikte hittades i bilden —
detta är det förväntade normalfallet, inte en brist. Vi gör semantiken
explicit med en boolesk flagga (`has_second_face`) istället för att
imputera NaN.

In [6]:
from src.preprocessing import flag_second_face

df = flag_second_face(df)
n_with_second_face = df["has_second_face"].sum()
pct_with_second_face = n_with_second_face / len(df) * 100

print(f"Rader med detekterat andra ansikte: {n_with_second_face} av {len(df)} ({pct_with_second_face:.2f}%)")

Rader med detekterat andra ansikte: 4096 av 62328 (6.57%)


### second_face_score: resultat

**4 096 av 62 328 rader (6,57%)** har ett detekterat andra ansikte i
bilden, vilket matchar siffran från steg 01. Detta bekräftar att
`second_face_score`-kolumnen är korrekt tolkad: NaN för de återstående
93,43% betyder "inget andra ansikte hittades" — det förväntade
normalfallet — inte en brist i datan.

`has_second_face`-flaggan gör denna semantik explicit och tydlig för
resten av pipelinen, utan att vi behöver imputera eller tolka NaN
kontextuellt varje gång kolumnen används.

Denna kolumn används inte direkt för filtrering i nuläget, men kan bli
relevant senare — t.ex. för att exkludera bilder med flera ansikten vid
ansiktsdetektering/embedding-extraktion, eftersom det då blir
tvetydigt vilket ansikte som ska användas.

## name: kartläggning av saknade värden

`name` används inte som feature eller målvariabel någonstans i
pipelinen — det är enbart en läsbar identifierare från Wikipedia-
metadatan. Vi bygger därför ingen dedikerad flagg-/filterfunktion i
src/preprocessing.py för detta fält (jämfört med gender och
face_detected, som faktiskt kommer filtreras vid modellträning);
det vore onödig infrastruktur för ett fält utan modellrelevans.
Vi dokumenterar bara omfattningen här.

In [7]:
n_missing_name = df["name"].isna().sum()
pct_missing_name = n_missing_name / len(df) * 100

print(f"Rader utan name: {n_missing_name} av {len(df)} ({pct_missing_name:.2f}%)")

Rader utan name: 124 av 62328 (0.20%)


## Ålder: kartläggning av orimliga värden

Vi flaggar rader med orimlig ålder (< 0 eller > 120 år) — ett känt
problem i IMDB-WIKI orsakat av felaktiga namn-till-person-matchningar
i Wikipedia-metadatan. Gränsen 120 år tillåter enstaka verifierat höga
åldrar (världens äldsta dokumenterade personer har levt till ~116–122
år) utan att vara godtyckligt tillåtande.

In [8]:
from src.preprocessing import flag_implausible_age

df = flag_implausible_age(df)
n_implausible = df["age_implausible"].sum()
pct_implausible = n_implausible / len(df) * 100

print(f"Rader med orimlig ålder: {n_implausible} av {len(df)} ({pct_implausible:.2f}%)")

Rader med orimlig ålder: 1498 av 62328 (2.40%)


In [9]:
df["age"].describe()

count    62328.000000
mean        37.053251
std         23.870392
min        -74.000000
25%         24.000000
50%         31.000000
75%         47.000000
max       1996.000000
Name: age, dtype: float64

### Ålder: resultat

**1 498 av 62 328 rader (2,40%)** flaggas med orimlig ålder (< 0 eller
> 120 år).

`describe()`-statistiken visar tydligt varför gränserna behövs:

- **min = -74** — ett foto daterat 74 år före personens angivna
  födelsedatum, ett extremfall av det kända namn-matchningsproblemet.
- **max = 1996** — ett orimligt extremvärde som sannolikt beror på en
  helt felaktig koppling mellan bild och Wikipedia-person.
- Resten av fördelningen ser rimlig ut: medelålder ~37 år, median 31 år,
  standardavvikelse ~24 år — vilket är typiskt för en dataset dominerad
  av porträttfoton av vuxna.

De 2,40% flaggade raderna bekräftar att `age_implausible`-flaggan fångar
ett litet men verkligt och betydelsefullt kvalitetsproblem i datasetet.
I linje med tidigare flaggor (`face_detected`, `gender_missing`)
filtreras dessa rader inte bort här, utan hanteras separat via
`filter_valid_age()`, tillämpad först där en tillförlitlig ålder
faktiskt krävs — t.ex. vid träning av åldersestimeringsmodellen.

## Sammanfattning: dataförberedelse och kvalitetsflaggor

Detta notebook-steg har kompletterat dataförberedelsen (steg 01) med:

1. **Ålder** (`age`): beräknad från `dob_matlab` och `photo_taken`.
2. **Ansiktsdetektering** (`face_detected`): 18 016 rader (28,91%) saknar
   detekterat ansikte.
3. **Gender** (`gender_missing`): 2 643 rader (4,24%) saknar kön.
4. **Andra ansikte** (`has_second_face`): 4 096 rader (6,57%) har ett
   andra ansikte i bilden — normalfall, ej brist.
5. **Name**: 124 rader (0,20%) saknar namn — dokumenterat, ingen
   dedikerad hanteringsfunktion (används ej i modellering).
6. **Orimlig ålder** (`age_implausible`): 1 498 rader (2,40%) flaggas,
   pga kända namn-till-person-matchningsfel i källdatan.

Samtliga flaggor bevarar rådata oförändrad och tar inte bort några
rader — filtrering sker separat och medvetet, först vid den punkt i
pipelinen (03–05) där respektive kvalitetsproblem faktiskt blir
relevant (t.ex. embedding-extraktion, målvariabel-träning).